# 06 · Sample description and transient colours (paper additions)

Numbers and figures for the paper's sample description, the comparison of objects with and without a SIMBAD
counterpart, and a colour–colour diagram of the transient light itself.

| Code | Figure |
|---|---|
| P1 | alerts per band; detections per DIA object; difference-flux magnitudes of the alerts; template magnitudes |
| P2 | Galactic latitude of DIA objects and SSO alerts |
| P3 | template magnitudes with / without a SIMBAD type (all fields, outside and inside bulge/plane); fractions with a SIMBAD type and with a Gaia DR3 source vs i_tmpl |
| T1 | r−i vs g−r from same-night difference fluxes, over the template colours |
| T2, T3 | the same transient colours by Fink CATS label and by SIMBAD group |

**Template colours vs transient colours.** Figures 1–3 and L1–L6 use template fluxes: the brightness of the static
sky at the object's position. T1–T3 instead use the difference fluxes (`psfFlux`), the light that changed. A colour
needs g, r and i close in time, because a transient changes, so T1–T3 only use alerts from the same night.

**Inputs:** the catalog and its pinned raw files, notebook 01's template table, notebook 02's science table (Gaia
names), notebook 04's exported colour table and the locus file. **Outputs:** `RUN_ROOT/06_paper_additions/` with the figures, `tables/` (per-object transient
colours, all object-nights, the per-figure counts) and `summary.json`. Run time on Arnor: a few minutes.

In [ ]:
# Shared configuration: paths, run folder, helpers (edit notebooks/science/_setup.py, not this cell).
from _setup import *
describe_config()
import pandas as pd
import alert_science_suite as suite
import paper_sample_figures as paper
template_photometry = require(RUN_ROOT / "01_template_colours/tables/dia_object_template_photometry.parquet", "01_template_colours")
science_table = require(RUN_ROOT / "02_science_suite/tables/dia_object_science_table.parquet", "02_science_suite")
export_table = require(RUN_ROOT / "04_stellar_locus/fink_dia_template_colours.csv.gz", "04_stellar_locus")
locus = require(LOCUS_FILE, "the locus file (see LOCUS_FILE in _setup.py)")
out = step_dir("06_paper_additions")
figs = out / "figures"
record, res = [], {}                  # figure records and results, filled below

## Step 1 · Load every alert and the per-object template table
Populations follow `analysis_contract_v1` and are asserted against the catalog's totals.

In [ ]:
files, qc, req = suite.load_catalog_inputs(CATALOG, DATA_ROOT)
src, counts = suite.prepare_sources(files, qc)
tp = pd.read_parquet(template_photometry)
print(f"{len(src):,} alerts", counts, f"   {len(tp):,} DIA objects")

## P · Sample description (P1, P2)
Difference-flux magnitudes use alerts with `psfFlux > 0`. Template magnitudes are the per-band medians of
`templateFlux` over each object's alerts, kept when positive with median S/N ≥ 5 (notebook 01).

In [ ]:
res["sample"] = paper.sample_summary(src, tp)
show([paper.fig_p1_sample(src, tp, figs, record), paper.fig_p2_galactic_latitude(src, tp, figs, record)])
res["sample"]

## P3 · Objects with and without a SIMBAD counterpart
Groups as in figure 3: a SIMBAD type from the latest alert whose crossmatch succeeded; no SIMBAD object within 1″;
or a crossmatch that failed on every alert (status unknown). The last panel shows how many of them have a Gaia DR3
source within 1″ in Fink's Gaia crossmatch (a lower limit where that crossmatch failed).

In [ ]:
sci = pd.read_parquet(science_table, columns=["dia_object_id", "gaia_name"])
show(paper.fig_p3_simbad_magnitudes(tp, sci, figs, record, res))
{k: v for k, v in res["P3_simbad_magnitudes"].items() if not k.startswith("fraction_")}

## T · Colours of the transient light (T1–T3)
DIA alerts with `psfFlux > 0` and `psfFlux / psfFluxErr ≥ 5` in g, r and i. Night = floor(MJD − 0.5). Per object,
night and band: the inverse-variance-weighted mean difference flux. Object-nights with all three bands are kept, and
each object contributes one night: the one whose weakest band has the highest S/N. Bulge and plane fields are left
out of the figures, because nothing is corrected for extinction.

In [ ]:
tc, object_nights = paper.transient_colours(src, tp)
tc.to_parquet(out / "tables" / "transient_colours_per_object.parquet", index=False)
object_nights.to_parquet(out / "tables" / "transient_colours_object_nights.parquet", index=False)
res["T_selection"] = {"object_nights_with_gri": int(len(object_nights)), "objects": int(len(tc)),
                      "nights_per_object_q50_max": [float(object_nights.groupby("oid").size().median()),
                                                    int(object_nights.groupby("oid").size().max())]}
show(paper.fig_t1_t3_transient_colours(tc, tp, locus, figs, record, res))
res["T_selection"], res["T_counts"]

## Counts for the paper figures
Objects per panel and per row for figures 2 and 3 (notebook 01, all fields) and L3 and L4 (notebook 04, outside the
bulge and plane fields).

In [ ]:
fig_counts = paper.paper_figure_counts(export_table, tp)
fig_counts.to_csv(out / "tables" / "paper_figure_counts.csv", index=False)
fig_counts

## Summary

In [ ]:
summary = {"run": RUN_ID, "step": "06_paper_additions", "code_sha": code_version(), "catalog": str(CATALOG),
           "catalog_sha256": suite.sha256(CATALOG), "template_photometry_sha256": suite.sha256(template_photometry),
           "science_table_sha256": suite.sha256(science_table), "export_table_sha256": suite.sha256(export_table), "locus_sha256": suite.sha256(locus), "raw_files": len(files),
           "packages": {m: __import__(m).__version__ for m in ("numpy", "pandas", "pyarrow", "matplotlib", "astropy", "duckdb")},
           "figures": record, "results": suite.jsonable(res), "runtime_s": round(time.time() - T0, 1)}
save_json(summary, out / "summary.json")
print(len(record), "figures written to", figs)